# DSC0152 - notebook v69

## Kenapa v69 ada

v68 menambah fold seed, tetapi public `v68_04` hanya **0.56411**, sedikit di bawah anchor `v66_04` **0.56424**. Kesimpulannya: menambah partisi saja tidak cukup.

v69 menguji tiga hal yang belum pernah diuji bersama:

1. **Factor features yang selama ini tidak terpakai.** v61-v68 menghitung `factor_z`, `factor_sd`, dan `fs_*` per fold, lalu tidak memasukkannya ke daftar kolom training. v69 memasukkannya.
2. **Blok sequence baru** (`cashflow`, `repayseq`, `prevseq`) yang mengukur perubahan, bukan level.
3. **Fold lebih banyak dan varian `lgb_lean` yang lebih kaku.** 15 fold, dua varian (`extra`, `stiff7`). Enam fold seed selesai; enam sisanya tidak dijalankan agar CPU bebas.

Tidak ada data eksternal, AutoML, atau AutoFE. Semua fitur historis dipotong pada `decision_datetime`; factor loading hanya memakai label fold training.


In [1]:
import os, sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, ks_2samp
from sklearn.metrics import roc_auc_score

RUN_TRAINING = False
FOLDS = 15
FOLD_SEEDS = '42,7,13,21,5,101,3,17,11,23,77,55'
LGB_SEEDS = '0'
VARIANTS = 'extra,stiff7'
DROP_PREFIXES = 'cf_,pv_'

HERE = Path.cwd().resolve()
ROOT = HERE
for cand in [HERE, *HERE.parents]:
    if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
        ROOT = cand
        break
AGAS = ROOT / 'agas'
VER = AGAS / 'v69'
OUT = VER / 'output'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(AGAS / 'shared'))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)


repo root   /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v69


## 1. Screening fitur

Semua baris memakai 5 fold dan fold seed 42. Tujuannya memilih blok dan konfigurasi, bukan menghasilkan submission. Kolom `factor` menunjukkan apakah enam kolom factor fold-safe ikut dilatih.


In [2]:
s = json.loads((OUT / 'summary_v69_screening.json').read_text())
rows = []
for name, d in s.items():
    rows.append({'run': name, 'auc': round(d['mean_auc'], 6),
                 'ks': round(d['mean_ks'], 6), 'config': d['config'],
                 'n_lean': d['n_lean'], 'factor': d['use_factor'],
                 'drops': ','.join(d['drops'])})
display(pd.DataFrame(rows).sort_values('auc', ascending=False))


,run,auc,ks,config,n_lean,factor,drops
6,repayseq_extra_factor,0.846747,0.524847,extra,526,True,"cf_,pv_"
7,repayseq_stiff7_factor,0.846368,0.527178,stiff7,526,True,"cf_,pv_"
2,all_blocks_factor,0.846204,0.525296,base,612,True,
4,repayseq_factor,0.846140,0.524248,base,526,True,"cf_,pv_"
1,base_factor,0.846035,0.524932,base,490,True,"cf_,rq_,pv_"
5,prevseq_factor,0.845989,0.524229,base,524,True,"cf_,rq_"
3,cashflow_factor,0.845928,0.524118,base,542,True,"rq_,pv_"
0,base_nofactor,0.845398,0.523272,base,490,False,"cf_,rq_,pv_"


## 2. Training v69

Hanya kaki `lgb_lean` yang dilatih ulang. Enam kaki v68 lainnya dipakai ulang; `make_submission_v69.py` mengganti `lgb_lean` di rantai blend yang sama. Partial disimpan per fold seed, jadi run bisa dilanjutkan. Run ini dihentikan setelah enam dari dua belas seed; agregasi tetap memakai semua seed yang selesai.


In [3]:
cmd = (f'python3 {TRAINING}/train_lean_v69.py '
       f'--features {OUT}/features_v69.pkl --variants {VARIANTS} '
       f'--folds {FOLDS} --fold-seeds {FOLD_SEEDS} --lgb-seeds {LGB_SEEDS} '
       f'--drop-prefixes {DROP_PREFIXES} --out-prefix v69')
print(cmd)
if RUN_TRAINING:
    subprocess.run(cmd, shell=True, check=True)


python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v69/training/train_lean_v69.py --features /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v69/output/features_v69.pkl --variants extra,stiff7 --folds 15 --fold-seeds 42,7,13,21,5,101,3,17,11,23,77,55 --lgb-seeds 0 --drop-prefixes cf_,pv_ --out-prefix v69


## 3. Kaki lean: v68 vs v69

`v68_lgb_lean` adalah resep lama tanpa factor features. `extra` dan `stiff7` adalah dua varian v69 dengan factor features dan blok `repayseq`. Angka v69 adalah rata-rata enam fold seed.


In [4]:
z68 = np.load(AGAS / 'v68/output/preds_v68.npz', allow_pickle=True)
z69 = np.load(OUT / 'preds_v69.npz', allow_pickle=True)
assert np.array_equal(z69['train_ids'], z68['train_ids'])
print('v69 seeds:', json.loads((OUT / 'summary_v69.json').read_text())['fold_seeds'])
y = z69['y'].astype(int)
r = lambda v: rankdata(v) / len(v)

def auc(p): return roc_auc_score(y, p)
def ks(p): return float(ks_2samp(p[y == 1], p[y == 0]).statistic)

rows = []
for name, o in [('v68_lgb_lean', z68['oof_lgb_lean']),
                ('v69_extra', z69['oof_extra']),
                ('v69_stiff7', z69['oof_stiff7'])]:
    rows.append({'leg': name, 'auc': round(auc(o), 6), 'ks': round(ks(o), 6)})
display(pd.DataFrame(rows))


v69 seeds: [42, 7, 13, 21, 5, 101]


,leg,auc,ks
0,v68_lgb_lean,0.846857,0.526881
1,v69_extra,0.848132,0.527736
2,v69_stiff7,0.847653,0.527933


## 4. Rantai blend

Rantai tidak berubah: `trees4 -> core -> tm20/tm30 -> ext0.5|tm30|lean`. Yang berubah hanya isi `lean`. Angka di bawah adalah OOF, bukan public.


In [5]:
ro68 = {k[4:]: r(z68[k]) for k in z68.files if k.startswith('oof_')}
rt68 = {k[5:]: r(z68[k]) for k in z68.files if k.startswith('test_')}
leans = ['extra', 'stiff7']
rows = []
for name in leans:
    lo, lt = r(z69[f'oof_{name}']), r(z69[f'test_{name}'])
    trees_o = (ro68['lgb_full'] + lo + ro68['cat_full'] + ro68['xgb_full']) / 4
    trees_t = (rt68['lgb_full'] + lt + rt68['cat_full'] + rt68['xgb_full']) / 4
    core_o = (2 * trees_o + ro68['lgbx_full']) / 3
    core_t = (2 * trees_t + rt68['lgbx_full']) / 3
    tm30_o = 0.7 * core_o + 0.3 * ro68['linear']
    tm30_t = 0.7 * core_t + 0.3 * rt68['linear']
    ext_o = r(tm30_o + 0.5 * (tm30_o - lo))
    ext_t = r(tm30_t + 0.5 * (tm30_t - lt))
    for tag, o in [('lean', lo), ('tm20', 0.8 * core_o + 0.2 * ro68['linear']),
                   ('tm30', tm30_o), ('ext0.5|tm30', ext_o)]:
        rows.append({'variant': name, 'chain': tag, 'auc': round(auc(o), 6), 'ks': round(ks(o), 6)})
display(pd.DataFrame(rows).sort_values(['variant', 'auc'], ascending=[True, False]))


,variant,chain,auc,ks
0,extra,lean,0.848132,0.527736
1,extra,tm20,0.847674,0.527580
2,extra,tm30,0.847543,0.528004
3,extra,ext0.5|tm30,0.846292,0.528254
4,stiff7,lean,0.847653,0.527933
5,stiff7,tm20,0.847609,0.527748
6,stiff7,tm30,0.847480,0.528223
7,stiff7,ext0.5|tm30,0.846470,0.527429


## 5. Kandidat dan validasi

Kandidat ditulis oleh `make_submission_v69.py`. Semua file harus 30000 baris, ID unik dan sama dengan test, serta probabilitas di `[0, 1]`.


In [6]:
cand = json.loads((OUT / 'summary_v69_candidates.json').read_text())
df = pd.DataFrame(cand['candidates'])
display(df[['slot', 'name', 'auc', 'ks', 'ks_minus_sigma', 'spearman_v66_anchor', 'valid']])
test_ids = pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id'])
for f in sorted(OUT.glob('submission_v69_*.csv')):
    d = pd.read_csv(f)
    assert len(d) == 30000 and d.application_id.is_unique
    assert set(d.application_id) == set(test_ids.application_id)
    assert d.default_probability.between(0, 1).all() and not d.default_probability.isna().any()
print(f'{len(list(OUT.glob("submission_v69_*.csv")))} submission files valid')


,slot,name,auc,ks,ks_minus_sigma,spearman_v66_anchor,valid
0,1,leg|extra,0.848132,0.527736,0.524136,0.991086,True
1,2,stack_logit,0.847997,0.527692,0.524092,0.996453,True
2,3,stack_anchor_w0.1,0.847913,0.528224,0.524624,0.998454,True
3,4,trees4|extra,0.847765,0.526741,0.523141,0.995292,True
4,5,tm20|extra,0.847674,0.527580,0.523980,0.998347,True
5,6,leg|stiff7,0.847653,0.527933,0.524333,0.991371,True
6,7,stack_anchor_w0.5,0.847566,0.528656,0.525056,0.999729,True
7,8,ext0.35|tm20|extra,0.847024,0.527747,0.524147,0.998898,True
8,9,ext0.65|tm20|stiff7,0.846480,0.526909,0.523309,0.998367,True
9,10,ext0.5|tm30|extra,0.846292,0.528254,0.524654,0.999465,True


11 submission files valid


## 6. Kesimpulan

v69 bukan penambahan seed lagi. Ada tiga perubahan yang bisa diukur:

- **Factor features** menaikkan AUC 5-fold dari 0.845398 ke 0.846035 (+0.000637) pada himpunan lean yang sama.
- **Blok `repayseq`** menambah sedikit lagi; `cashflow` dan `prevseq` tidak lolos screening.
- **Konfigurasi `extra`** adalah varian terbaik di screening (0.846747 dengan `repayseq` + factor), di atas `stiff7` 0.846368 pada fold seed yang sama.

Pada enam seed yang selesai, bagged `extra` = **0.848132 / KS 0.527736**, di atas `v68_lgb_lean` 0.846857 / 0.526881 (+0.001275 AUC). Kandidat teratas adalah `leg|extra` (AUC 0.848132, Spearman 0.991086 terhadap anchor), lalu `stack_logit`, `stack_anchor_w0.1`, dan `trees4|extra`.

Yang belum diketahui tetap public score. OOF yang lebih tinggi bukan jaminan public, terutama karena noise public sekitar 0.0053. Kandidat yang dikirim harus dibandingkan dengan anchor `0.56424`, bukan hanya dengan v68.
